# AI Stock Prediction - XGBoost Training
**Mục tiêu:** Huấn luyện model XGBoost không bị Data Leakage. Bổ sung tính năng VNINDEX để đánh giá xu hướng thị trường chung.

In [ ]:
# === BƯỚC 1: KẾT NỐI GOOGLE DRIVE VÀ LOAD DATA ===
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
import pickle, json, os

df = pd.read_csv('/content/drive/MyDrive/StockData/all_stocks_train.csv')
df['trading_date'] = pd.to_datetime(df['trading_date'])
df = df.sort_values(['symbol', 'trading_date']).reset_index(drop=True)

print(f"Raw data: {len(df):,} dòng, {df['symbol'].nunique()} mã")

In [ ]:
# === BƯỚC 2A: LẤY DỮ LIỆU VNINDEX TRƯỚC KHI FILTER ===
# (Lưu ý: Phải tách VNINDEX ở đây vì bước sau sẽ filter mất mã VNINDEX)
vnindex = df[df['symbol'] == 'VNINDEX'][['trading_date', 'close']].copy()

if len(vnindex) > 0:
    print(f"Tìm thấy VNINDEX: {len(vnindex)} ngày")
    vnindex = vnindex.sort_values('trading_date')
    vnindex['vnindex_momentum_5']  = vnindex['close'].pct_change(5).clip(-0.3, 0.3)
    vnindex['vnindex_momentum_20'] = vnindex['close'].pct_change(20).clip(-0.3, 0.3)
    
    vnindex_diff = vnindex['close'].diff()
    gain = vnindex_diff.clip(lower=0).ewm(alpha=1/14, adjust=False).mean()
    loss = (-vnindex_diff.clip(upper=0)).ewm(alpha=1/14, adjust=False).mean()
    rs = gain / loss.replace(0, np.nan)
    vnindex['vnindex_rsi'] = 100 - (100 / (1 + rs))
    
    HAS_VNINDEX = True
else:
    print("Không có VNINDEX trong data → bỏ qua")
    HAS_VNINDEX = False

In [ ]:
# === BƯỚC 2B: FILTER MÃ CHẤT LƯỢNG VÀ MERGE VNINDEX ===
df = df[df['exchange'].isin(['HOSE', 'HNX'])]
df = df[df['volume_sma_20'] > 100000]

if HAS_VNINDEX:
    df = df.merge(
        vnindex[['trading_date', 'vnindex_momentum_5', 'vnindex_momentum_20', 'vnindex_rsi']],
        on='trading_date', how='left'
    )
    print("Đã merge tính năng VNINDEX vào bộ dữ liệu chính.")

print(f"Sau filter exchange + volume: {len(df):,} dòng, {df['symbol'].nunique()} mã")

In [ ]:
# === BƯỚC 3: ĐỊNH NGHĨA FEATURES & LÀM SẠCH ===
FEATURES = [
    'price_vs_sma20', 'price_vs_sma50', 'sma20_vs_sma50',
    'price_momentum_5', 'price_momentum_10', 'price_momentum_20',
    'macd_norm', 'macd_hist_norm',
    'adx_14', 'di_plus', 'di_minus',
    'rsi_14', 'stoch_k', 'stoch_d', 'williams_r',
    'bb_pct_b', 'bb_width_norm',
    'atr_pct',
    'cmf_20', 'volume_ratio', 'volume_momentum_5',
    'candle_body_pct', 'candle_upper_pct', 'candle_lower_pct',
    'is_doji', 'is_hammer',
    'is_bull_engulfing', 'is_bear_engulfing',
    'is_morning_star', 'is_evening_star'
]

if HAS_VNINDEX:
    FEATURES += ['vnindex_momentum_5', 'vnindex_momentum_20', 'vnindex_rsi']
    print(f"Tổng features: {len(FEATURES)} (CÓ tính năng thị trường chung VNINDEX)")
else:
    print(f"Tổng features: {len(FEATURES)} (KHÔNG có tính năng VNINDEX)")

TARGET = 'direction_5d'

df = df.dropna(subset=[TARGET])
df[TARGET] = df[TARGET].astype(int)

# Forward fill
df[FEATURES] = df.groupby('symbol')[FEATURES].ffill()

# Bỏ qua các dòng bị thiếu (ví dụ: ngày đầu tiên của VNINDEX rsi có thể NaN)
df = df.dropna(subset=FEATURES)

print(f"Sau khi làm sạch dữ liệu: {len(df):,} dòng, {df['symbol'].nunique()} mã")

In [ ]:
# === BƯỚC 4: TÁCH TRAIN/TEST KHÔNG SHUFFLE ===
CUTOFF = '2025-10-01'
train = df[df['trading_date'] <  CUTOFF]
test  = df[df['trading_date'] >= CUTOFF]

X_train = train[FEATURES]
y_train = train[TARGET]
X_test  = test[FEATURES]
y_test  = test[TARGET]

print(f"\nTrain: {len(X_train):,} dòng ({train['trading_date'].min().date()} -> {train['trading_date'].max().date()})")
print(f"Test : {len(X_test):,} dòng ({test['trading_date'].min().date()} -> {test['trading_date'].max().date()})")

train_up_ratio = y_train.mean()
test_up_ratio = y_test.mean()
print(f"\nTỷ lệ tăng trong Train: {train_up_ratio:.2%}")
print(f"Tỷ lệ tăng trong Test : {test_up_ratio:.2%}")

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"-> scale_pos_weight tính từ Train = {scale_pos_weight:.4f}")

In [ ]:
# === BƯỚC 5: TRAIN XGBOOST VỚI BỘ THAM SỐ CHỐNG NHIỄU (REGULARIZED) ===
model = xgb.XGBClassifier(
    n_estimators          = 2000,
    learning_rate         = 0.01,
    max_depth             = 4,
    subsample             = 0.6,
    colsample_bytree      = 0.6,
    min_child_weight      = 30,
    gamma                 = 1.0,
    reg_alpha             = 0.5,
    reg_lambda            = 2.0,
    scale_pos_weight      = scale_pos_weight,
    eval_metric           = 'auc',
    early_stopping_rounds = 100,
    random_state          = 42,
    n_jobs                = -1,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=200
)

In [ ]:
# === BƯỚC 6: ĐÁNH GIÁ ===
y_pred      = model.predict(X_test)
y_pred_prob = model.predict_proba(X_test)[:, 1]

print("\n" + "="*60)
print("KẾT QUẢ ĐÁNH GIÁ (TEST SET)")
print("="*60)
print(f"Accuracy : {accuracy_score(y_test, y_pred):.2%}")
print(f"ROC-AUC  : {roc_auc_score(y_test, y_pred_prob):.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Giảm', 'Tăng']))

# 1. Win rate theo từng ngưỡng
print("\nWin rate tại các ngưỡng:")
print(f"{'Ngưỡng':>8} {'Số tín hiệu':>12} {'Win rate':>10}")
print("-" * 32)
test_eval = test.copy()
test_eval['score'] = y_pred_prob
for threshold in [0.50, 0.55, 0.60, 0.65, 0.70]:
    signals = test_eval[test_eval['score'] >= threshold]
    if len(signals) == 0:
        continue
    win_rate = (signals[TARGET] == 1).mean()
    print(f"{threshold:>8.2f} {len(signals):>12,} {win_rate:>10.2%}")

# 2. Phân bố score
print("\nPhân bố điểm dự báo:")
total_test = len(test_eval)
for thresh in [0.60, 0.65]:
    count = len(test_eval[test_eval['score'] >= thresh])
    pct = count / total_test
    print(f"Số mã có score >= {thresh:.2f}: {count:,} ({pct:.2%})")

# 3. Feature importance top 15
fi = pd.DataFrame({
    'feature':    FEATURES,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 15 features quan trọng nhất:")
print(fi.head(15).to_string(index=False))

plt.figure(figsize=(10, 8))
plt.barh(fi.head(15)['feature'][::-1], fi.head(15)['importance'][::-1])
plt.title('Top 15 Feature Importance')
plt.tight_layout()
plt.show()

In [ ]:
# === BƯỚC 7: LƯU MODEL ===
OUTPUT_DIR = '/content/drive/MyDrive/StockData/models'
os.makedirs(OUTPUT_DIR, exist_ok=True)

with open(f'{OUTPUT_DIR}/xgb_direction_5d.pkl', 'wb') as f:
    pickle.dump(model, f)

with open(f'{OUTPUT_DIR}/features.json', 'w') as f:
    json.dump({'features': FEATURES}, f, indent=2)

fi.to_csv(f'{OUTPUT_DIR}/feature_importance.csv', index=False)

print("✅ Đã lưu thành công model và features.json!")